In [30]:
import json
from typing import Any, Optional
from pathlib import Path

In [31]:
import json
from typing import Any, Optional, Union
from urllib.parse import urlparse, parse_qs


def should_include_entry(url: str, content_type: Optional[str] = None) -> bool:
    """Decide if this entry is worth keeping based on URL and content type"""
    if not url:
        return False

    # Skip common static assets that don't affect flow
    static_extensions = [
        ".css",
        ".js",
        ".png",
        ".jpg",
        ".jpeg",
        ".gif",
        ".svg",
        ".ico",
        ".woff",
        ".woff2",
        ".ttf",
        ".eot",
        ".map",
        ".webp",
        ".avif",
    ]

    url_lower = url.lower()

    # Skip if it's a static file
    if any(url_lower.endswith(ext) for ext in static_extensions):
        return False

    # Skip if it's a versioned JS/CSS file (e.g., ?v=1.0.1)
    if "?v=" in url_lower and any(ext in url_lower for ext in [".js", ".css"]):
        return False

    # Skip static content types
    if content_type:
        content_type_lower = content_type.lower()
        skip_types = [
            "text/css",
            "image/",
            "font/",
            "application/javascript",
            "application/x-javascript",
            "text/javascript",
        ]
        if any(stype in content_type_lower for stype in skip_types):
            return False

    return True


def get_content_type(headers: list[dict]) -> Optional[str]:
    """Extract Content-Type from headers list"""
    for header in headers:
        if header.get("name", "").lower() == "content-type":
            return header.get("value")
    return None


def extract_important_headers(headers: list[dict], is_request: bool = True) -> dict:
    """Extract only the most important headers for understanding data flow"""
    important_request_headers = {
        "authorization",
        "cookie",
        "content-type",
        "accept",
        "x-csrf-token",
        "x-api-key",
        "referer",
        "origin",
    }

    important_response_headers = {
        "content-type",
        "location",
        "set-cookie",
        "cache-control",
        "x-request-id",
        "x-correlation-id",
    }

    important = important_request_headers if is_request else important_response_headers

    result = {}
    for header in headers:
        name = header.get("name", "").lower()
        if name in important or name.startswith("x-") and "amz" not in name:
            # Simplify cookie values to just show presence
            if name in ["cookie", "set-cookie"]:
                result[name] = "[cookies present]"
            else:
                result[name] = header.get("value")

    return result


def parse_url_details(url: str) -> dict:
    """Parse URL to extract endpoint and parameters"""
    parsed = urlparse(url)

    # Extract path segments that look like IDs or parameters
    path_parts = [p for p in parsed.path.split("/") if p]

    # Try to identify if any path parts are IDs (numbers, UUIDs, etc.)
    path_params = []
    clean_path_parts = []
    for part in path_parts:
        # Check if it looks like an ID
        if part.isdigit() or len(part) > 20:  # Likely an ID
            path_params.append(part)
            clean_path_parts.append("{id}")
        else:
            clean_path_parts.append(part)

    return {
        "endpoint": "/" + "/".join(clean_path_parts) if clean_path_parts else "/",
        "path_params": path_params if path_params else None,
        "query_params": parse_qs(parsed.query) if parsed.query else None,
        "host": parsed.netloc,
    }


def extract_form_data(post_data: dict) -> Optional[dict]:
    """Extract and simplify form/POST data"""
    if not post_data:
        return None

    mime_type = post_data.get("mimeType", "")
    text = post_data.get("text", "")

    if not text:
        return None

    result = {"type": mime_type}

    # Parse different content types
    if "application/json" in mime_type:
        try:
            data = json.loads(text)
            # Simplify nested structures
            result["data"] = simplify_json_structure(data)
        except:
            result["data"] = "[invalid json]"
    elif "application/x-www-form-urlencoded" in mime_type:
        try:
            result["data"] = parse_qs(text)
        except:
            result["data"] = text[:200] + "..." if len(text) > 200 else text
    else:
        # For other types, just show a preview
        result["data"] = text[:200] + "..." if len(text) > 200 else text

    return result


def simplify_json_structure(
    obj: Any, max_depth: int = 2, current_depth: int = 0
) -> Any:
    """Simplify JSON structure to show keys and basic types"""
    if current_depth >= max_depth:
        return (
            "[nested object]"
            if isinstance(obj, dict)
            else "[nested array]"
            if isinstance(obj, list)
            else obj
        )

    if isinstance(obj, dict):
        return {
            k: simplify_json_structure(v, max_depth, current_depth + 1)
            for k, v in obj.items()
        }
    elif isinstance(obj, list):
        if len(obj) > 0:
            # Show first item structure and count
            return f"[{len(obj)} items like: {simplify_json_structure(obj[0], max_depth, current_depth + 1)}]"
        return []
    elif isinstance(obj, str) and len(obj) > 50:
        return obj[:50] + "..."
    return obj


def analyze_response_content(response: dict) -> Optional[dict]:
    """Analyze response content for important data"""
    content = response.get("content", {})
    mime_type = content.get("mimeType", "")

    # Only analyze HTML and JSON responses
    if "html" not in mime_type and "json" not in mime_type:
        return None

    size = content.get("size", -1)

    result = {
        "type": mime_type.split(";")[0] if ";" in mime_type else mime_type,
        "size": size,
    }

    # Note if content is compressed
    if content.get("compression"):
        result["compressed"] = True

    # Reference to actual content file if present
    if content.get("_sha1"):
        result["content_ref"] = content["_sha1"]

    return result


def clean_trace_entry(entry: dict[str, Any]) -> Optional[dict[str, Any]]:
    """
    Clean a single trace entry focusing on API flow analysis
    Returns None if entry should be skipped entirely
    """
    if entry.get("type") != "resource-snapshot" or not entry.get("snapshot"):
        return None

    snapshot = entry["snapshot"]
    request = snapshot.get("request", {})
    response = snapshot.get("response", {})

    # Check if we should include this entry
    url = request.get("url", "")
    content_type = get_content_type(response.get("headers", []))

    if not should_include_entry(url, content_type):
        return None

    # Parse URL for better understanding
    url_details = parse_url_details(url)

    # Build focused entry for API flow analysis
    cleaned = {
        "timestamp": snapshot.get("startedDateTime"),
        "duration_ms": snapshot.get("time"),
        "endpoint": url_details["endpoint"],
        "host": url_details["host"],
        "method": request.get("method"),
        "status": response.get("status"),
    }

    # Add path parameters if present
    if url_details["path_params"]:
        cleaned["path_params"] = url_details["path_params"]

    # Add query parameters if present
    if url_details["query_params"]:
        cleaned["query_params"] = url_details["query_params"]

    # Add important request headers
    req_headers = extract_important_headers(request.get("headers", []), is_request=True)
    if req_headers:
        cleaned["request_headers"] = req_headers

    # Add POST data if present
    form_data = extract_form_data(request.get("postData"))
    if form_data:
        cleaned["request_body"] = form_data

    # Add important response headers
    resp_headers = extract_important_headers(
        response.get("headers", []), is_request=False
    )
    if resp_headers:
        cleaned["response_headers"] = resp_headers

    # Analyze response content
    response_analysis = analyze_response_content(response)
    if response_analysis:
        cleaned["response_info"] = response_analysis

    # Add redirect info if present
    if response.get("status") in [301, 302, 303, 307, 308]:
        for header in response.get("headers", []):
            if header.get("name", "").lower() == "location":
                cleaned["redirect_to"] = header.get("value")
                break

    return cleaned


def group_by_endpoint(entries: list[dict]) -> dict:
    """Group entries by endpoint to see patterns"""
    grouped = {}
    for entry in entries:
        endpoint = entry.get("endpoint", "unknown")
        if endpoint not in grouped:
            grouped[endpoint] = []
        grouped[endpoint].append(entry)
    return grouped


def analyze_flow_patterns(entries: list[dict]) -> dict:
    """Analyze entries to identify common patterns and flows"""
    analysis = {
        "total_requests": len(entries),
        "unique_endpoints": len(set(e.get("endpoint") for e in entries)),
        "methods_used": list(set(e.get("method") for e in entries)),
        "status_codes": {},
        "endpoints_with_params": [],
        "redirects": [],
        "api_calls": [],  # Non-HTML endpoints
        "form_submissions": [],
    }

    for entry in entries:
        # Count status codes
        status = str(entry.get("status", "unknown"))
        analysis["status_codes"][status] = analysis["status_codes"].get(status, 0) + 1

        # Track endpoints with parameters
        if entry.get("query_params") or entry.get("path_params"):
            endpoint_info = {
                "endpoint": entry.get("endpoint"),
                "method": entry.get("method"),
                "query_params": list(entry.get("query_params", {}).keys()),
                "path_params": bool(entry.get("path_params")),
            }
            if endpoint_info not in analysis["endpoints_with_params"]:
                analysis["endpoints_with_params"].append(endpoint_info)

        # Track redirects
        if entry.get("redirect_to"):
            analysis["redirects"].append(
                {
                    "from": entry.get("endpoint"),
                    "to": entry.get("redirect_to"),
                    "status": entry.get("status"),
                }
            )

        # Track API calls (non-HTML responses)
        response_info = entry.get("response_info", {})
        if response_info and "html" not in response_info.get("type", ""):
            analysis["api_calls"].append(
                {
                    "endpoint": entry.get("endpoint"),
                    "method": entry.get("method"),
                    "response_type": response_info.get("type"),
                    "params": list(entry.get("query_params", {}).keys()),
                }
            )

        # Track form submissions
        if entry.get("request_body"):
            analysis["form_submissions"].append(
                {
                    "endpoint": entry.get("endpoint"),
                    "method": entry.get("method"),
                    "content_type": entry.get("request_body", {}).get("type"),
                }
            )

    return analysis


def process_trace_file(
    input_file: str, output_file: Optional[str] = None, include_analysis: bool = True
) -> dict:
    """
    Process trace file focusing on API flow analysis

    Args:
        input_file: Path to input trace file
        output_file: Path to output file (optional)
        include_analysis: Whether to include flow analysis

    Returns:
        dict with cleaned entries and optional analysis
    """
    # Load entries
    har_entries = []
    with open(input_file, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                try:
                    har_entry = json.loads(line)
                    har_entries.append(har_entry)
                except json.JSONDecodeError as e:
                    print(f"Skipping malformed JSON line: {e}")

    print(f"Loaded {len(har_entries)} entries from {input_file}")

    # Clean each entry
    cleaned_entries = []
    for entry in har_entries:
        cleaned = clean_trace_entry(entry)
        if cleaned:
            cleaned_entries.append(cleaned)

    print(f"Cleaned down to {len(cleaned_entries)} relevant entries")

    result = {"entries": cleaned_entries}

    # Add analysis if requested
    if include_analysis:
        result["analysis"] = analyze_flow_patterns(cleaned_entries)
        result["grouped_by_endpoint"] = group_by_endpoint(cleaned_entries)

    # Save to file if specified
    if output_file:
        with open(output_file, "w", encoding="utf-8") as f:
            json.dump(result, f, indent=2)
        print(f"Saved cleaned data to {output_file}")

    return result

In [32]:
def clean_all_trace_networks(base_path="../../../data/traces"):
    base_dir = Path(base_path)

    if not base_dir.exists():
        print(f"Base directory {base_path} does not exist")
        return

    # Get all subdirectories in the base path
    council_dirs = [d for d in base_dir.iterdir() if d.is_dir()]
    for council_dir in council_dirs:
        trace_network_file = council_dir / "traces" / "trace.network"
        if trace_network_file.exists():
            clean_trace = process_trace_file(
                trace_network_file,
                output_file=council_dir / "trace.json",
                include_analysis=True,
            )
         

In [33]:
clean_all_trace_networks()

Loaded 381 entries from ../../../data/traces/north_kesteven_district_council/traces/trace.network
Cleaned down to 56 relevant entries
Saved cleaned data to ../../../data/traces/north_kesteven_district_council/trace.json
Loaded 105 entries from ../../../data/traces/mid_sussex_district_council/traces/trace.network
Cleaned down to 22 relevant entries
Saved cleaned data to ../../../data/traces/mid_sussex_district_council/trace.json
Loaded 38 entries from ../../../data/traces/buckinghamshire_council/traces/trace.network
Cleaned down to 11 relevant entries
Saved cleaned data to ../../../data/traces/buckinghamshire_council/trace.json
Loaded 160 entries from ../../../data/traces/north_west_leicestershire_district_council/traces/trace.network
Cleaned down to 18 relevant entries
Saved cleaned data to ../../../data/traces/north_west_leicestershire_district_council/trace.json
Loaded 148 entries from ../../../data/traces/warrington_borough_council/traces/trace.network
Cleaned down to 37 relevant en